# Component selection, corrected (fixes the notebook-06 RMSECV graph)

Notebook 06's per-variant RMSECV graph is **misleading**, though the underlying PLS math is
fine and the CAL-0 minimum (k≈21) is robust. Four issues, fixed here:

| # | Problem in 06 | Fix |
|---|---------------|-----|
| 1 | 5-fold *shuffled* CV → jagged curves | **10-fold interleaved** CV (smoother; matched R exactly) |
| 2 | absolute-µg RMSECV overlaid → not comparable across variants | **%RMSECV** (÷ subset mean EC) |
| 3 | diagnostics (CAL-2/3/4) dominate the plot | candidates vs diagnostics in **separate panels** |
| 4 | 2% tol in the table but 5% in the 2nd-deriv section (21 vs 13) | **one tolerance** everywhere |

**This notebook does not change the variant definitions** — it rebuilds the *identical*
CAL-0…CAL-5 masks from notebook 06 and only corrects the component analysis on top.

In [1]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.cross_decomposition import PLSRegression
from scipy.signal import savgol_filter

PRED = Path("../spartan_ec_2026_06_16")
Path("figures").mkdir(exist_ok=True); Path("tables").mkdir(exist_ok=True)

Xdf = pd.read_csv(PRED / "data/rds_EC_X.csv")
y = pd.read_csv(PRED / "data/rds_EC_Ymeasured.csv")["Y_measured"].to_numpy()
X = Xdf[[c for c in Xdf.columns if c != "id"]].to_numpy(float)
WN = pd.read_csv(PRED / "data/rds_EC_coef_k18.csv")["wavenumber"].to_numpy()
assert len(WN) == X.shape[1]
print("EC set:", X.shape, "| EC range", round(y.min(), 2), "-", round(y.max(), 2),
      "| samples EC>150:", int((y > 150).sum()), "(these few dominate absolute RMSE)")

EC set: (906, 2722) | EC range 0.77 - 393.33 | samples EC>150: 7 (these few dominate absolute RMSE)


## Rebuild the identical CAL-0…CAL-5 masks (verbatim from notebook 06)

`BASELINE_K = 21` is notebook 06's `k0` (its `first_major_min` of the CAL-0 curve); the
baseline model at that k defines the cleaned / removed / below-1:1 sets. Reproducing it here
keeps the variant sets byte-for-byte identical to 06 (906 / 860 / 46 / 469 / 22 / 480).

In [2]:
BASELINE_K = 21
base = PLSRegression(n_components=BASELINE_K, scale=False).fit(X, y)
pred0 = base.predict(X).ravel()

keep = np.ones(len(y), bool)              # CAL-1: iterative 3-sigma trim at BASELINE_K
for _ in range(3):
    m = PLSRegression(n_components=BASELINE_K, scale=False).fit(X[keep], y[keep])
    rr = y - m.predict(X).ravel()
    newkeep = np.abs(rr) <= 3 * rr[keep].std()
    if (newkeep == keep).all():
        break
    keep = keep & newkeep

ETH_LOW, ETH_HIGH = 10.0, 100.0           # CAL-5 placeholder band (unchanged from 06)
MASKS = {
    "CAL-0 all-nofilter": np.ones(len(y), bool),
    "CAL-1 cleaned":      keep,
    "CAL-2 removed-only": ~keep,
    "CAL-3 below-1:1":    pred0 < y,
    "CAL-4 EC-high>=70":  y >= 70.0,
    "CAL-5 Eth-range":    (y >= ETH_LOW) & (y <= ETH_HIGH),
}
CANDIDATES  = ["CAL-0 all-nofilter", "CAL-1 cleaned", "CAL-5 Eth-range"]
DIAGNOSTICS = ["CAL-2 removed-only", "CAL-3 below-1:1", "CAL-4 EC-high>=70"]
for k, m in MASKS.items():
    print(f"{k:22s} n={int(m.sum()):4d}   mean EC={y[m].mean():6.1f} µg")

CAL-0 all-nofilter     n= 906   mean EC=  16.9 µg
CAL-1 cleaned          n= 860   mean EC=  13.7 µg
CAL-2 removed-only     n=  46   mean EC=  75.8 µg
CAL-3 below-1:1        n= 469   mean EC=  18.3 µg
CAL-4 EC-high>=70      n=  22   mean EC= 144.3 µg
CAL-5 Eth-range        n= 480   mean EC=  21.3 µg


## Corrected CV — 10-fold interleaved, per-component reconstruction

Interleaved folds (`i % 10`) match the app's R `pls` scheme; per-component predictions are
reconstructed from one fit per fold (so the whole curve costs 10 fits, not 10×kmax). `maxc`
is capped below the smallest training-fold size so small variants don't fit degenerate PLS.

In [3]:
def pred_all_ncomps(m, Xnew, mc):
    Xc = Xnew - m._x_mean
    out = np.empty((Xnew.shape[0], mc))
    for k in range(1, mc + 1):
        b = m.x_rotations_[:, :k] @ m.y_loadings_[:, :k].T
        out[:, k - 1] = (Xc @ b).ravel() + m._y_mean
    return out

def cv_rmsecv(Xs, ys, folds=10, maxc=30):
    f = np.arange(len(ys)) % folds
    mc = int(min(maxc, min((f != i).sum() for i in range(folds)) - 1, Xs.shape[1]))
    press = np.zeros(mc)
    for i in range(folds):
        tr, te = f != i, f == i
        m = PLSRegression(n_components=mc, scale=False).fit(Xs[tr], ys[tr])
        press += np.sum((pred_all_ncomps(m, Xs[te], mc) - ys[te][:, None])**2, axis=0)
    ks = np.arange(1, mc + 1)
    return ks, np.sqrt(press / len(ys))

TOL = 0.05                                  # ONE tolerance, used everywhere
def pick_k(ks, rmsecv, tol=TOL):
    return int(ks[np.argmax(rmsecv <= rmsecv.min() * (1 + tol))])

curves = {}
for name, mask in MASKS.items():
    ks, rc = cv_rmsecv(X[mask], y[mask])
    pct = 100 * rc / y[mask].mean()
    curves[name] = dict(ks=ks, rmsecv=rc, pct=pct, k=pick_k(ks, rc),
                        n=int(mask.sum()), meanEC=float(y[mask].mean()))
    print(f"{name:22s} k*={curves[name]['k']:2d}  RMSECV_min={rc.min():6.2f} µg  "
          f"%RMSECV_min={pct.min():5.1f}%")

CAL-0 all-nofilter     k*=19  RMSECV_min= 16.49 µg  %RMSECV_min= 97.7%


CAL-1 cleaned          k*= 1  RMSECV_min= 19.67 µg  %RMSECV_min=143.3%
CAL-2 removed-only     k*= 1  RMSECV_min= 66.64 µg  %RMSECV_min= 87.9%


CAL-3 below-1:1        k*= 4  RMSECV_min= 37.83 µg  %RMSECV_min=206.2%
CAL-4 EC-high>=70      k*= 1  RMSECV_min= 84.58 µg  %RMSECV_min= 58.6%


CAL-5 Eth-range        k*=13  RMSECV_min=  7.19 µg  %RMSECV_min= 33.8%


## Figure 1 — %RMSECV, candidates vs. diagnostics (the corrected graph)

Left: the three variants that are actual calibration candidates (CAL-0/1/5). Right: the
small, unstable inverse/threshold diagnostics (CAL-2/3/4) — shown but kept out of the
candidates' scale. Dashed line = the k picked by the single 5%-tolerance rule.

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True)
for ax, group, title in [(axes[0], CANDIDATES, "Candidate calibrations"),
                         (axes[1], DIAGNOSTICS, "Diagnostics (small n — unstable)")]:
    for name in group:
        c = curves[name]
        ax.plot(c["ks"], c["pct"], "-o", ms=3, lw=1.2,
                label=f"{name}  (n={c['n']}, k*={c['k']})")
        ax.axvline(c["k"], ls=":", lw=0.8, color=ax.get_lines()[-1].get_color())
    ax.set_title(title); ax.set_xlabel("n PLS components")
    ax.set_ylabel("%RMSECV  (= RMSECV ÷ subset mean EC)")
    ax.legend(fontsize=8)
fig.suptitle("Corrected component selection — %RMSECV, 10-fold interleaved CV", y=1.02, fontsize=13)
plt.tight_layout(); plt.savefig("figures/fig09_component_selection_pct.png", dpi=140, bbox_inches="tight")
print("saved figures/fig09_component_selection_pct.png"); plt.show()

saved figures/fig09_component_selection_pct.png


/var/folders/7k/65ckdzsj0w3171qv80rh8dgr0000gn/T/ipykernel_73819/3890987231.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  print("saved figures/fig09_component_selection_pct.png"); plt.show()


## Figure 2 — raw vs. 2nd-derivative (CAL-0): why the raw curve is jagged

The raw curve wobbles (low components model the PTFE baseline, and a handful of extreme-EC
samples dominate the error). The **2nd-derivative** curve is far smoother and near its floor
from very few components — the Weakley argument for selecting k on 2nd-derivative spectra.

In [5]:
X2 = savgol_filter(X, window_length=11, polyorder=2, deriv=2, axis=1)
ks_raw, rc_raw = cv_rmsecv(X, y)
ks_d2,  rc_d2  = cv_rmsecv(X2, y)
pct_raw = 100 * rc_raw / y.mean(); pct_d2 = 100 * rc_d2 / y.mean()
k_raw, k_d2 = pick_k(ks_raw, rc_raw), pick_k(ks_d2, rc_d2)

fig, ax = plt.subplots(figsize=(8.5, 5))
ax.plot(ks_raw, pct_raw, "-o", ms=3, color="#1f77b4", label=f"raw  (k*={k_raw})")
ax.plot(ks_d2,  pct_d2,  "-o", ms=3, color="#d62728", label=f"2nd-derivative  (k*={k_d2})")
ax.axvline(k_raw, ls=":", lw=1, color="#1f77b4"); ax.axvline(k_d2, ls=":", lw=1, color="#d62728")
ax.set_xlabel("n PLS components"); ax.set_ylabel("%RMSECV (CAL-0, n=906)")
ax.set_title("CAL-0: raw spectra are jagged; 2nd-derivative is smooth (one 5% tol → consistent k)")
ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig("figures/fig09_raw_vs_d2.png", dpi=140, bbox_inches="tight")
print(f"raw k*={k_raw} (min {rc_raw.min():.2f} µg) | 2nd-deriv k*={k_d2} (min {rc_d2.min():.2f} µg)")
print("saved figures/fig09_raw_vs_d2.png"); plt.show()

raw k*=19 (min 16.49 µg) | 2nd-deriv k*=9 (min 16.97 µg)
saved figures/fig09_raw_vs_d2.png


/var/folders/7k/65ckdzsj0w3171qv80rh8dgr0000gn/T/ipykernel_73819/581097604.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  print("saved figures/fig09_raw_vs_d2.png"); plt.show()


## Corrected summary table

Compare on **%RMSECV** (comparable across variants), with the single-rule `k*`. Contrast
with notebook 06's `k_firstmin` (2% tol on the jagged 5-fold curve) to see what changed.

In [6]:
old = pd.read_csv("tables/component_selection_by_variant.csv").set_index("name")
rows = []
for name, c in curves.items():
    rows.append({"name": name, "n": c["n"], "mean_EC": round(c["meanEC"], 1),
                 "k*_corrected": c["k"], "RMSECV_min_ug": round(float(c["rmsecv"].min()), 2),
                 "pctRMSECV_min": round(float(c["pct"].min()), 1),
                 "k_firstmin_nb06": int(old.loc[name, "k_firstmin"]) if name in old.index else None})
tbl = pd.DataFrame(rows)
tbl.to_csv("tables/component_selection_corrected.csv", index=False)
print(tbl.to_string(index=False))
print("\nwrote tables/component_selection_corrected.csv")

              name   n  mean_EC  k*_corrected  RMSECV_min_ug  pctRMSECV_min  k_firstmin_nb06
CAL-0 all-nofilter 906     16.9            19          16.49           97.7               21
     CAL-1 cleaned 860     13.7             1          19.67          143.3                1
CAL-2 removed-only  46     75.8             1          66.64           87.9                1
   CAL-3 below-1:1 469     18.3             4          37.83          206.2                4
 CAL-4 EC-high>=70  22    144.3             1          84.58           58.6                6
   CAL-5 Eth-range 480     21.3            13           7.19           33.8               15

wrote tables/component_selection_corrected.csv


### What changed vs. notebook 06
- The **honest comparison is %RMSECV** — on it, CAL-0 (keep-everything) and CAL-5 are the
  only stable candidates; CAL-2/3/4 have large %RMSECV (small-n instability), confirming they
  are diagnostics, not calibrations.
- `k*` here uses **one** 5% tolerance on the smoother 10-fold curve; it no longer flips
  between 21 and 13 depending on which section you read.
- **Adopt 2nd-derivative preprocessing before finalizing k** — Figure 2 shows the raw curve's
  jaggedness is a preprocessing artifact, not real model structure.
- Variant definitions are unchanged; only the component-selection analysis is corrected.